In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from nsetools import Nse

In [2]:
def get_data(stock):
    df = yf.download(stock, interval="5m", period="5d")

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index(drop=True)
    df = df.dropna()

    return df

In [3]:
def add_features(df):
    df = df.copy()

    for col in ['Close','Volume','High','Low']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['rsi'] = ta.momentum.RSIIndicator(df['Close']).rsi()
    df['ma20'] = df['Close'].rolling(20).mean()
    df['ma50'] = df['Close'].rolling(50).mean()
    df['volume_ma'] = df['Volume'].rolling(20).mean()

    typical_price = (df['High'] + df['Low'] + df['Close']) / 3
    df['vwap'] = (typical_price * df['Volume']).cumsum() / df['Volume'].cumsum()

    return df

In [4]:
def create_target(df):
    df = df.copy()

    close = pd.Series(df['Close'].values, index=df.index)
    future_price = close.shift(-3)

    df['target'] = (future_price > close).astype(int)

    return df

In [5]:
df = get_data("TATASTEEL.NS")
df = add_features(df)
df = create_target(df)

df = df.dropna().reset_index(drop=True)

features = ['rsi','ma20','ma50','volume_ma']

df = df.dropna(subset=features + ['target'])

X = df[features]
y = df['target']

X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

model = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
model.fit(X_train, y_train)

print("Model Accuracy:", model.score(X_test, y_test))

[*********************100%***********************]  1 of 1 completed


Model Accuracy: 0.48484848484848486


In [6]:
def strategy_filter(df):
    latest = df.iloc[-1]

    return (
        (latest['Close'] > latest['vwap']) and
        (latest['Volume'] > latest['volume_ma']) and
        (latest['Close'] > df['High'].rolling(20).max().iloc[-1])
    )


def generate_trade(df, model):
    latest = df.iloc[-1]

    features = latest[['rsi','ma20','ma50','volume_ma']].fillna(0)

    prob = model.predict_proba(features.values.reshape(1, -1))[0][1]

    if prob > 0.65 and strategy_filter(df):
        return "BUY", prob
    elif prob < 0.35:
        return "SELL", prob

    return "NO TRADE", prob

In [7]:
nifty50 = [
    "RELIANCE.NS","TCS.NS","INFY.NS","HDFCBANK.NS","ICICIBANK.NS",
    "KOTAKBANK.NS","SBIN.NS","BHARTIARTL.NS","ITC.NS","HINDUNILVR.NS",
    "LT.NS","AXISBANK.NS","ASIANPAINT.NS","MARUTI.NS","SUNPHARMA.NS",
    "TITAN.NS","ULTRACEMCO.NS","NESTLEIND.NS","WIPRO.NS","HCLTECH.NS",
    "BAJFINANCE.NS","BAJAJFINSV.NS","POWERGRID.NS","NTPC.NS","ONGC.NS",
    "COALINDIA.NS","JSWSTEEL.NS","TATASTEEL.NS","INDUSINDBK.NS","TECHM.NS",
    "ADANIENT.NS","ADANIPORTS.NS","BPCL.NS","BRITANNIA.NS","CIPLA.NS",
    "DIVISLAB.NS","DRREDDY.NS","EICHERMOT.NS","GRASIM.NS","HEROMOTOCO.NS",
    "HINDALCO.NS","IOC.NS","M&M.NS","SHREECEM.NS","TATAMOTORS.NS",
    "UPL.NS","BAJAJ-AUTO.NS","SBILIFE.NS","HDFCLIFE.NS","ICICIPRULI.NS"
]

In [8]:
nse = Nse()

gainers = nse.get_top_gainers()
losers = nse.get_top_losers()

gainer_symbols = [s['symbol'] + ".NS" for s in gainers]
loser_symbols = [s['symbol'] + ".NS" for s in losers]

movers = list(set(gainer_symbols + loser_symbols))

In [9]:
final_stocks = list(set(nifty50 + movers))
final_stocks = list(set(nifty50 + movers))

In [10]:
results = []

for stock in final_stocks:
    try:
        temp_df = get_data(stock)
        temp_df = add_features(temp_df)
        temp_df = create_target(temp_df)
        temp_df = temp_df.dropna().reset_index(drop=True)

        signal, prob = generate_trade(temp_df, model)

        results.append({
            "stock": stock,
            "signal": signal,
            "confidence": round(prob, 2)
        })

        print(f"{stock} done ✅")

    except Exception as e:
        print(f"{stock} error ❌", e)

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")


TATAMOTORS.NS error ❌ single positional indexer is out-of-bounds


[*********************100%***********************]  1 of 1 completed
C:\Users\Lenovo\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


ICICIBANK.NS done ✅


[*********************100%***********************]  1 of 1 completed
C:\Users\Lenovo\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


...

[*********************100%***********************]  1 of 1 completed
C:\Users\Lenovo\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


WIPRO.NS done ✅


[*********************100%***********************]  1 of 1 completed

DRREDDY.NS done ✅



C:\Users\Lenovo\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [11]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(by="confidence", ascending=False)

results_df

,stock,signal,confidence
39,RELIANCE.NS,NO TRADE,0.54
0,ICICIBANK.NS,NO TRADE,0.50
33,BEL.NS,NO TRADE,0.46
21,POWERGRID.NS,NO TRADE,0.45
10,TATASTEEL.NS,NO TRADE,0.45
3,IOC.NS,NO TRADE,0.41
8,ITC.NS,NO TRADE,0.40
53,WIPRO.NS,NO TRADE,0.38
36,BPCL.NS,NO TRADE,0.36
48,BAJFINANCE.NS,SELL,0.34


In [12]:
best_trades = results_df[
    (results_df["signal"] == "SELL" ) & (results_df["confidence"] > 0.2)
]

best_trades

,stock,signal,confidence
48,BAJFINANCE.NS,SELL,0.34
52,HDFCBANK.NS,SELL,0.31
4,SHRIRAMFIN.NS,SELL,0.31
14,KOTAKBANK.NS,SELL,0.31
49,AXISBANK.NS,SELL,0.28
9,TRENT.NS,SELL,0.28
17,TECHM.NS,SELL,0.28
38,DIVISLAB.NS,SELL,0.28
37,TCS.NS,SELL,0.28
32,BAJAJFINSV.NS,SELL,0.24


In [13]:
results_df.to_csv("results.csv", index=False)